In [ ]:
# 1. sort out data with label "MDD" and "HEALTHY"

import pandas as pd
from pathlib import Path

# data & label path
data_dir = Path("E:/preprocessed_data_high gamma")  
label_path = Path("E:\TDBRAIN_participants_V2.xlsx")

# obtain existing id
existing_ids = [p.name for p in data_dir.iterdir() if p.is_dir()]

# obtain label from Excel
df = pd.read_excel(label_path)

# save data with formal_status = MDD or HEALTHY 
df_filtered = df[df["participants_ID"].isin(existing_ids)]
df_filtered = df_filtered[df_filtered["formal_status"].isin(["MDD", "HEALTHY"])]

# print the number of the samples
print("Total number of eligible labeled samples:", len(df_filtered))
print("MDD samples:", (df_filtered["formal_status"] == "MDD").sum())
print("HEALTHY samples:", (df_filtered["formal_status"] == "HEALTHY").sum())

# Construct an ID -> label mapping dictionary
label_dict = dict(zip(df_filtered["participants_ID"], df_filtered["formal_status"]))


In [ ]:
# 2. load EEG data -> EA + Within-subject normalization (preserve temporal sequence)

import os
import mne
import numpy as np
import scipy
from tqdm import tqdm

# define Euclidean Alignment + Within-subject normalization
def EA_whitten(data):
    aligned_data = []
    for trials in data:
        covs = [np.cov(trial) for trial in trials]
        mean_cov = np.mean(covs, axis=0)
        mean_cov_inv_sqrt = np.linalg.pinv(scipy.linalg.sqrtm(mean_cov))

        aligned_trials = []
        for trial in trials:
            trial_aligned = mean_cov_inv_sqrt @ trial
            trial_aligned = (trial_aligned - np.mean(trial_aligned, axis=1, keepdims=True)) / \
                            np.std(trial_aligned, axis=1, keepdims=True)
            aligned_trials.append(trial_aligned)
        aligned_data.append(aligned_trials)
    return aligned_data


X_all = []
y_all = []
subject_ids = []


for subj_id, label in tqdm(label_dict.items(), desc="Loading EEG data"):
    subj_path = data_dir / subj_id
    subj_epochs = []

    # traverse from ses-1 to ses-4
    for ses_num in range(1, 5):
        ses_path = subj_path / f"ses-{ses_num}" / "eeg" / "preprocessed-epo.fif"
        if ses_path.exists():
            try:
                epochs = mne.read_epochs(str(ses_path), preload=True, verbose=False)
                data = epochs.get_data()  # shape: (n_trials, n_channels, n_times)
                subj_epochs.extend([trial for trial in data])
            except Exception as e:
                print(f"loading fail: {ses_path}, error: {e}")
                continue

    if subj_epochs:   # only keep subjects with valid data
        X_all.append(subj_epochs)
        y_all.append(0 if label == "HEALTHY" else 1)
        subject_ids.append(subj_id)

print(f"Number of successfully read subjects: {len(X_all)}")

X_all = EA_whitten(X_all)



In [ ]:
# 3. Expand each subject's epoch to construct X, y, subj_ids

X = []
y = []
subj_ids = []

for subj_data, label, subj_id in zip(X_all, y_all, subject_ids):
    X.extend(subj_data) 
    y.extend([label] * len(subj_data))  
    subj_ids.extend([subj_id] * len(subj_data))  


X = np.array(X)
y = np.array(y)
subj_ids = np.array(subj_ids)


print("total trials number：", X.shape[0])
print("dimension (n_trials, n_channels, n_times)：", X.shape)

In [ ]:
# 4. Construct training/validation sets by category (HEALTHY 80/20, MDD equal amount for training) 

import numpy as np
from sklearn.model_selection import train_test_split


indices_healthy = np.where(y == 0)[0]
indices_mdd = np.where(y == 1)[0]

X_healthy = X[indices_healthy]
y_healthy = y[indices_healthy]

X_mdd = X[indices_mdd]
y_mdd = y[indices_mdd]


X_healthy_train, X_healthy_val, y_healthy_train, y_healthy_val = train_test_split(
    X_healthy, y_healthy, test_size=0.2, random_state=42, stratify=y_healthy
)

n_healthy_train = X_healthy_train.shape[0]
print("HEALTHY training datast:", n_healthy_train)

# For MDD, instead of an 80/20 split, randomly sample n_healthy_train from all MDD samples as training data, and use the rest as validation data.
np.random.seed(42)
indices_perm = np.random.permutation(X_mdd.shape[0])
X_mdd_train = X_mdd[indices_perm[:n_healthy_train]]
y_mdd_train = y_mdd[indices_perm[:n_healthy_train]]
X_mdd_val = X_mdd[indices_perm[n_healthy_train:]]
y_mdd_val = y_mdd[indices_perm[n_healthy_train:]]

# Construct a balanced training set: The number of HEALTHY and MDD training samples are both n_healthy_train
X_train = np.concatenate([X_healthy_train, X_mdd_train], axis=0)
y_train = np.concatenate([y_healthy_train, y_mdd_train], axis=0)

# Construct validation set: Merge HEALTHY validation set with the remaining MDD validation samples
X_val = np.concatenate([X_healthy_val, X_mdd_val], axis=0)
y_val = np.concatenate([y_healthy_val, y_mdd_val], axis=0)

print(f"number of the traning dataset: {X_train.shape[0]}, HEALTHY: {(y_train==0).sum()}, MDD: {(y_train==1).sum()}")
print(f"number of the validation dataset: {X_val.shape[0]}, HEALTHY: {(y_val==0).sum()}, MDD: {(y_val==1).sum()}")


In [ ]:
# 5. PyTorch Dataset & DataLoader


from torch.utils.data import Dataset, DataLoader
import torch
import random

class EEGCSPDataset(Dataset):
    def __init__(self, X, X_csp, y):
        self.data   = self.normalize(X)
        self.labels = y
    def normalize(self, data):
        normalized = np.empty_like(data, dtype=np.float32)
        for i in range(data.shape[0]):
            sample = data[i]
            mean = sample.mean(axis=1, keepdims=True)
            std  = sample.std(axis=1, keepdims=True) + 1e-6
            normalized[i] = (sample - mean) / std
        return normalized
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self, idx):
        X_sample   = torch.tensor(self.data[idx], dtype=torch.float32)
        X_csp_sample = torch.zeros(1, dtype=torch.float32)
        y_sample     = torch.tensor(self.labels[idx], dtype=torch.long)
        return X_sample, X_csp_sample, y_sample



BATCH_SIZE = 32
train_dataset = EEGCSPDataset(X_train, None, y_train)
val_dataset   = EEGCSPDataset(X_val,   None, y_val)

def seed_worker(worker_id):
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, worker_init_fn=seed_worker, generator=g)

print("DataLoader completed")


In [ ]:
# 6. define M-ShallowConvNet
import torch
import torch.nn as nn
import torch.nn.functional as F

class MShallowNet(nn.Module):
    def __init__(self, in_channels, n_times, n_classes):
        super().__init__()
        self.conv_time  = nn.Conv2d(1, 16, kernel_size=(1, 10), stride=(1, 2))
        self.bn_time    = nn.BatchNorm2d(16)
        self.drop_time  = nn.Dropout(0.5)
        self.conv_space = nn.Conv2d(16, 32, kernel_size=(in_channels, 1))
        self.bn_space   = nn.BatchNorm2d(32)
        self.drop_space = nn.Dropout(0.5)
        self.pool       = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(32, n_classes)

    def forward(self, x, x_csp=None):
        # x: [B, n_channels, n_times]
        x = x.unsqueeze(1)                              # [B,1,C,T]
        x = F.elu(self.bn_time(self.conv_time(x)))
        x = self.drop_time(x)
        x = F.elu(self.bn_space(self.conv_space(x)))
        x = self.drop_space(x)
        x = torch.square(x)
        x = self.pool(x)                                # [B,32,1,1]
        x = torch.log(x + 1e-6)                         # [B,32,1,1]
        x = x.view(x.size(0), -1)                       # [B,32]
        return self.fc(x)                               # [B,n_classes]


In [ ]:
# 7. training set + Focal Loss + L2 
from torch.optim import Adam
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, input, target):
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

# Get the dimensions of the raw EEG data
in_channels = X_train.shape[1]
n_times = X_train.shape[2]
n_classes = len(np.unique(y_train))

# Initialize the model
model = MShallowNet(in_channels=in_channels, n_times=n_times, n_classes=n_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Adam + L2 
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Calculate class weights for the training set to prevent data imbalance from affecting training
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

criterion = FocalLoss(gamma=2.0, weight=class_weights)


In [ ]:
# 8. training

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    recall_score,
    confusion_matrix
)


def set_full_determinism(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_full_determinism(42)

# define FocalLoss
class FocalLoss(nn.Module):
    def __init__(self, alpha=2, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction=='mean' else focal_loss


def init_model(in_channels, n_times, n_classes):
    model = MShallowNet(in_channels, n_times, n_classes).to(device)
    def weights_init(m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    model.apply(weights_init)
    return model


in_channels = X_train.shape[1]     # EEG channel number
n_times = X_train.shape[2]         # EEG sequence length
n_classes = len(np.unique(y_train))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = init_model(in_channels, n_times, n_classes).to(device)

classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

criterion_focal = FocalLoss(alpha=2, gamma=2, reduction='mean')
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 200
train_losses = []
val_accuracies = []
val_sensitivities = []
val_specificities = []
val_f1s           = []
val_aucs          = []

for epoch in range(1, num_epochs+1):
    model.train()
    total_loss, total_samples = 0.0, 0
    for xb, x_csp, yb in train_loader:
        xb, x_csp, yb = xb.to(device), x_csp.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb, x_csp)
        loss    = criterion_focal(outputs, yb)
        loss.backward()
        optimizer.step()
        total_loss   += loss.item() * xb.size(0)
        total_samples+= xb.size(0)
    epoch_loss = total_loss / total_samples
    train_losses.append(epoch_loss)
    
    # val
    model.eval()
    y_true, y_pred, y_proba = [], [], []
    with torch.no_grad():
        for xb, x_csp, yb in val_loader:
            xb, x_csp = xb.to(device), x_csp.to(device)
            logits   = model(xb, x_csp)
            probs    = F.softmax(logits, dim=1)[:,1].cpu().numpy()  
            preds    = logits.argmax(dim=1).cpu().numpy()
            yb_np    = yb.cpu().numpy()
            
            y_true.extend(yb_np.tolist())
            y_pred.extend(preds.tolist())
            y_proba.extend(probs.tolist())
    
    # Accuracy
    acc = (np.array(y_pred) == np.array(y_true)).mean()
    val_accuracies.append(acc)
    
    # Sensitivity (Recall for positive class)
    sens = recall_score(y_true, y_pred, pos_label=1)
    val_sensitivities.append(sens)
    # Specificity = TN / (TN + FP)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp)
    val_specificities.append(spec)
    # F1
    f1 = f1_score(y_true, y_pred, pos_label=1)
    val_f1s.append(f1)
    # AUC
    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = float('nan')
    val_aucs.append(auc)
    
    
    print(
        f"Epoch {epoch}/{num_epochs}  "
        f"Loss: {epoch_loss:.4f},  Acc: {acc:.4f},  "
        f"Sens: {sens:.4f},  Spec: {spec:.4f},  "
        f"F1: {f1:.4f},  AUC: {auc:.4f}"
    )

# saved to .csv
results = {
    'epoch':      list(range(1, num_epochs+1)),
    'train_loss': train_losses,
    'val_acc':    val_accuracies,
    'sensitivity':val_sensitivities,
    'specificity':val_specificities,
    'f1_score':   val_f1s,
    'auc':        val_aucs
}
df = pd.DataFrame(results)

df.to_csv('training_mshallow high gamma.csv', index=False, encoding='utf-8-sig')

print("all results saved")
print("training completed")



In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter  

matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False


smooth_train_losses = savgol_filter(train_losses, window_length=11, polyorder=3)
smooth_val_accuracies = savgol_filter(val_accuracies, window_length=11, polyorder=3)

plt.figure(figsize=(12, 5))

# training
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), smooth_train_losses, color='b', label='smoothed')
plt.plot(range(1, num_epochs + 1), train_losses, color='b', alpha=0.3, linestyle='--', label='original')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Train Loss', fontsize=12)
plt.title('Traing Loss Curve', fontsize=14)
plt.legend()
plt.grid(True)

# val
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), smooth_val_accuracies, color='g', label='smoothed')
plt.plot(range(1, num_epochs + 1), val_accuracies, color='g', alpha=0.3, linestyle='--', label='original')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Validation Accuracy Curve', fontsize=14)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()